# 05 — RAPTOR Tree Exploration

Visualises the RAPTOR hierarchical summarisation tree:
- Cluster assignments with UMAP 2-D projections
- Section-level and paper-level summary nodes
- Dendrogram-like tree structure
- Embedding similarity heatmap across levels

In [ ]:
import asyncio
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize

from production_rag.core.config import get_settings
from production_rag.core.llm_client import get_llm_client
from production_rag.core.logging import setup_logging
from production_rag.core.types import ChunkLevel
from production_rag.ingestion.chunkers.factory import ChunkStrategy, get_chunker
from production_rag.ingestion.embedder import get_embedder
from production_rag.ingestion.loaders.arxiv_loader import ArXivLoader
from production_rag.ingestion.raptor.tree_builder import RAPTORTreeBuilder

setup_logging(json_logs=False)
settings = get_settings()
embedder = get_embedder(settings)
llm = get_llm_client(settings)

ARXIV_ID = "2312.10997"  # RAG survey paper — good for RAPTOR demo

async def load_and_chunk():
    loader = ArXivLoader()
    docs = await loader.load([ARXIV_ID])
    chunker = get_chunker(ChunkStrategy.RECURSIVE, settings)
    chunks = []
    for doc in docs:
        chunks.extend(chunker.chunk(doc))
    return docs, chunks

docs, leaf_chunks = asyncio.run(load_and_chunk())
print(f"Loaded {len(docs)} doc(s), {len(leaf_chunks)} leaf chunks")

In [ ]:
# Embed leaf chunks
async def embed_chunks(chunks):
    texts = [c.chunk_text for c in chunks]
    return await embedder.embed_documents(texts)

leaf_embeddings = asyncio.run(embed_chunks(leaf_chunks))
print(f"Embedded {len(leaf_embeddings)} chunks — dim={len(leaf_embeddings[0])}")

In [ ]:
# Build RAPTOR tree
from production_rag.ingestion.raptor.tree_builder import RAPTORTreeBuilder

builder = RAPTORTreeBuilder(embedder=embedder, llm=llm, settings=settings)
all_chunks = asyncio.run(builder.build(leaf_chunks, leaf_embeddings))

by_level = {}
for c in all_chunks:
    by_level.setdefault(c.chunk_level.value, []).append(c)

print("\nChunks per level:")
for k, v in sorted(by_level.items()):
    print(f"  {k}: {len(v)} chunks")

In [ ]:
# UMAP 2D projection coloured by level
try:
    import umap
    E = np.array([c.embedding for c in all_chunks if c.embedding])
    levels = [c.chunk_level.value for c in all_chunks if c.embedding]
    level_map = {"leaf": 0, "section": 1, "paper": 2}

    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15)
    coords = reducer.fit_transform(normalize(E))

    fig, ax = plt.subplots(figsize=(10, 7))
    colors = ["#3498db", "#e67e22", "#e74c3c"]
    for lvl, col in zip(["leaf", "section", "paper"], colors):
        mask = np.array([l == lvl for l in levels])
        ax.scatter(coords[mask, 0], coords[mask, 1], c=col, label=lvl,
                   alpha=0.7, s=20 if lvl == "leaf" else 80, edgecolors="white" if lvl != "leaf" else None)
    ax.legend(title="Level")
    ax.set_title(f"RAPTOR Tree — UMAP projection ({ARXIV_ID})")
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    plt.tight_layout()
    plt.savefig("../eval_results/raptor_umap.png", dpi=150)
    plt.show()
except ImportError:
    print("pip install umap-learn to enable UMAP visualisation")

In [ ]:
# Print first section-level summary
section_chunks = by_level.get("section", [])
if section_chunks:
    print("=== First section summary ===")
    print(section_chunks[0].chunk_text[:800])